In [5]:
pip install plantcv

  Using cached plantcv-4.2.1-py3-none-any.whl (334 kB)
  Using cached altair-5.0.1-py3-none-any.whl (471 kB)
ERROR: Could not find a version that satisfies the requirement xarray>=2022.11.0 (from plantcv) (from versions: 0.7.0, 0.7.1, 0.7.2, 0.8.0rc1, 0.8.0, 0.8.1, 0.8.2, 0.9.0rc1, 0.9.0, 0.9.1, 0.9.2, 0.9.3, 0.9.4, 0.9.5, 0.9.6, 0.10.0rc1, 0.10.0rc2, 0.10.0, 0.10.1, 0.10.2, 0.10.3, 0.10.4, 0.10.5, 0.10.6, 0.10.7, 0.10.8, 0.10.9, 0.11.0, 0.11.1, 0.11.2, 0.11.3, 0.12.0, 0.12.1, 0.12.2, 0.12.3, 0.13.0, 0.14.0, 0.14.1, 0.15.0, 0.15.1, 0.16.0, 0.16.1, 0.16.2, 0.17.0, 0.18.0, 0.18.1, 0.18.2, 0.19.0, 0.20.0, 0.20.1, 0.20.2)
ERROR: No matching distribution found for xarray>=2022.11.0 (from plantcv)
Note: you may need to restart the kernel to use updated packages.


In [16]:
import matplotlib.pyplot as plt
import os
%matplotlib inline
from plantcv import plantcv as pcv
import cv2
import numpy as np
import pandas as pd
import math
import numbers
import time
from plantcv.plantcv import params
from plantcv.plantcv import outputs
from plantcv.plantcv import color_palette
from plantcv.plantcv.morphology import find_tips
from plantcv.plantcv.morphology import segment_path_length
from plantcv.plantcv.morphology import segment_euclidean_length
from plantcv.plantcv._debug import _debug
from plantcv.plantcv._helpers import _cv2_findcontours, _object_composition

In [17]:
def rotate(img, rotation_deg, crop, cx, cy):
    """
    Rotate an image by a specified angle.

    Parameters:
    ----------
    img : numpy.ndarray
        RGB or grayscale image data.
    rotation_deg : float
        Rotation angle in degrees. Positive values rotate counter-clockwise.
    crop : bool
        If True, the output image size will match the input image.
        If False, the image will be resized to fit the rotated content.
    cx : float
        X-coordinate of the center of rotation.
    cy : float
        Y-coordinate of the center of rotation.

    Returns:
    -------
    rotated_img : numpy.ndarray
        Rotated image.
    """
    iy, ix = img.shape[:2]
    m = cv2.getRotationMatrix2D((cx, cy), rotation_deg, 1.0)
    cos = np.abs(m[0, 0])
    sin = np.abs(m[0, 1])

    if not crop:
        # compute new bounding dimensions
        nw = int((iy * sin) + (ix * cos))
        nh = int((iy * cos) + (ix * sin))

        # update translation terms in the rotation matrix
        m[0, 2] += (nw / 2) - (ix / 2)
        m[1, 2] += (nh / 2) - (iy / 2)

        rotated_img = cv2.warpAffine(img, m, (nw, nh), flags=cv2.INTER_CUBIC)
    else:
        rotated_img = cv2.warpAffine(img, m, (ix, iy), flags=cv2.INTER_CUBIC)

    # Optional: Debug image output (only if debugging is active)
    _debug(
        visual=rotated_img,
        filename=os.path.join(params.debug_outdir, f'{params.device}_rotated_img.png')
    )

    return rotated_img

def shorty_axis_pseudolandmarks(img, mask, label=None):
    """
    Generate 20 pseudo-landmarks along the vertical axis of a binary spine mask.

    Parameters:
    ----------
    img : numpy.ndarray
        Original grayscale or RGB image.
    mask : numpy.ndarray
        Binary mask (white object on black background).
    label : str, optional
        Label used to name observations. Defaults to params.sample_label.

    Returns:
    -------
    center_h : np.ndarray
        Array of shape (20, 1, 2) representing x,y coordinates of the landmarks.
    """
    if label is None:
        label = params.sample_label

    # Get the external contour of the mask
    cnt, cnt_str = _cv2_findcontours(bin_img=mask)
    obj = _object_composition(contours=cnt, hierarchy=cnt_str)

    if not np.any(obj):
        return ('NA', 'NA'), ('NA', 'NA'), ('NA', 'NA')

    x, y, width, height = cv2.boundingRect(obj)
    extent = height

    if extent < 22:
        raise IndexError("Spine height too small for landmarking (<22 pixels).")

    inc = int(extent / 22)  # creates 21 intervals → 20 usable pseudo-landmarks
    point_range = [(y + inc * (i - 1), y + inc * i) for i in range(1, 23)]

    x_centroids = []
    y_centroids = []

    for low_point, high_point in point_range:
        rows = []
        vals = list(range(low_point, high_point))

        for v in vals:
            value = obj[v == obj[:, 0, 1]]
            if len(value) > 0:
                largest = value[:, 0, 0].max()
                smallest = value[:, 0, 0].min()
                rows.append(largest - smallest)

        yval = int((high_point + low_point) / 2)
        y_centroids.append(yval)

        window = np.copy(mask)
        window[:low_point] = 0
        window[high_point:] = 0
        s = cv2.moments(window)

        if len(rows) > 0:
            if s['m00'] > 0.001:
                smx = s['m10'] / s['m00']
            else:
                smx = s['m10'] / 0.001  # Avoid divide by zero
            x_centroids.append(int(smx))
        else:
            x_centroids.append(int((largest + smallest) / 2))

    # Discard first and last point → use middle 20 points
    x_centroids = x_centroids[1:-1]
    y_centroids = y_centroids[1:-1]

    center_h = np.array(list(zip(x_centroids, y_centroids)))
    center_h.shape = (20, 1, 2)

    # Draw on the image
    img2 = np.copy(img)
    for pt in center_h:
        cv2.circle(img2, (int(pt[0, 0]), int(pt[0, 1])), params.line_thickness, (0, 79, 255), -1)

    _debug(
        visual=img2,
        filename=os.path.join(params.debug_outdir, f"{params.device}_y_axis_pseudolandmarks.png")
    )

    # Record landmark coordinates
    center_h_list = [pt[0].tolist() for pt in center_h]
    outputs.add_observation(
        sample=label,
        variable='center_h_lmk',
        trait='center horizontal landmark coordinates',
        method='plantcv.plantcv.x_axis_pseudolandmarks',
        scale='none',
        datatype=tuple,
        value=tuple(center_h_list),
        label='none'
    )

    return center_h


def gpty_axis_pseudolandmarks(img, mask, label=None):
    """
    Divide up object contour into 55 equidistant segments and generate landmarks for each.

    Inputs:
    img      = Copy of the original image; landmarks are drawn on this if debug is True.
    mask     = Binary mask of object (white = object, black = background).
    label    = Optional label for observation output (default = pcv.params.sample_label).

    Returns:
    center_h = List of 50 landmark coordinates within the center (excluding outermost 5).
    """
    # Use default label if none provided
    if label is None:
        label = params.sample_label

    # Find object contours in binary mask
    cnt, cnt_str = _cv2_findcontours(bin_img=mask)

    # Merge contours into a single object
    obj = _object_composition(contours=cnt, hierarchy=cnt_str)

    # Return 'NA' if object is empty (e.g., no object found)
    if not np.any(obj):
        return ('NA', 'NA'), ('NA', 'NA'), ('NA', 'NA')

    # Get bounding box and height
    x, y, width, height = cv2.boundingRect(obj)
    extent = height

    center_h = []        # Final landmark coordinates
    center_h_list = []   # Landmark coordinates in list format for output

    # If object is tall enough for 55 slices (1% increments)
    if extent >= 55:
        inc = extent / 55  # Interval height

        pts_max = []  # Upper bounds of intervals
        pts_min = []  # Lower bounds of intervals

        # Determine vertical slice bounds
        for i in range(1, 56):
            if i == 1:
                pt_max = y
                pt_min = y + (inc * i)
            else:
                pt_max = y + (inc * (i - 1))
                pt_min = y + (inc * i)
            pts_max.append(pt_max)
            pts_min.append(pt_min)

        point_range = list(zip(pts_max, pts_min))

        x_centroids = []
        y_centroids = []

        # Process each interval to compute horizontal landmarks
        for pt in point_range:
            low_point, high_point = pt
            rows = []

            vals = list(range(int(low_point), int(high_point)))

            for v in vals:
                # Find contour coordinates matching the current row (y = v)
                value = obj[v == obj[:, 0, 1]]
                if len(value) > 0:
                    largest = value[:, 0, 0].max()
                    smallest = value[:, 0, 0].min()
                    row_width = largest - smallest
                    rows.append(row_width)

            yval = int((high_point + low_point) / 2)
            y_centroids.append(yval)

            # Isolate region of interest in mask
            window = np.copy(mask)
            window[:int(low_point)] = 0
            window[int(high_point):] = 0

            s = cv2.moments(window)

            if len(rows) > 0:
                row_width = sum(rows) / len(rows)
                if s['m00'] > 0.001:
                    smx = s['m10'] / s['m00']
                else:
                    smx = s['m10'] / 0.001  # Prevent divide-by-zero
                x_centroids.append(int(smx))
            else:
                row_width = 1
                smx = (largest + smallest) / 2  # fallback if no rows found
                x_centroids.append(int(smx))

        # Drop outer slices (first 3 and last 2 intervals)
        x_centroids = x_centroids[3:-2]
        y_centroids = y_centroids[3:-2]

        center_h = list(zip(x_centroids, y_centroids))
        center_h = np.array(center_h)
        center_h.shape = (50, 1, 2)

        # Draw landmarks on copy of original image
        img2 = np.copy(img)
        for i in center_h:
            x = i[0, 0]
            y = i[0, 1]
            cv2.circle(img2, (int(x), int(y)), params.line_thickness, (0, 79, 255), -1)

        _debug(
            visual=img2,
            filename=os.path.join(params.debug_outdir, f"{params.device}_y_axis_pseudolandmarks.png")
        )

    else:
        raise IndexError("Index out of range")  # Object too short for pseudolandmarks

    # Save landmarks to output
    for pt in center_h:
        center_h_list.append(pt[0].tolist())

    outputs.add_observation(
        sample=label,
        variable='center_h_lmk',
        trait='center horizontal landmark coordinates',
        method='plantcv.plantcv.x_axis_pseudolandmarks',
        scale='none',
        datatype=tuple,
        value=tuple(center_h_list),
        label='none'
    )

    return center_h



def segment_curvature(segmented_img, objects, label=None):
    """
    Calculate curvature of segments based on geodesic vs Euclidean length (2D tortuosity).

    Inputs:
    segmented_img = Image with segmented regions
    objects       = List of contours
    label         = Optional label (default = pcv.params.sample_label)

    Returns:
    curvature_measure[0] = Curvature score of the first object
    spineLength          = Geodesic (path) length of the first object
    straightLength       = Euclidean length of the first object
    """
    if label is None:
        label = params.sample_label

    label_coord_x = []
    label_coord_y = []
    labeled_img = segmented_img.copy()

    # Temporarily disable debug to suppress unwanted output
    debug = params.debug
    params.debug = None

    # Measure segment lengths
    _ = segment_euclidean_length(segmented_img, objects, label="backend")
    _ = segment_path_length(segmented_img, objects, label="backend")

    eu_lengths = outputs.observations['backend']['segment_eu_length']['value']
    path_lengths = outputs.observations['backend']['segment_path_length']['value']

    # Tortuosity = path / euclidean length
    curvature_measure = [float(x / y) for x, y in zip(path_lengths, eu_lengths)]

    # Generate color palette
    rand_color = color_palette(num=len(objects), saved=True)

    # Draw line between endpoints of each segment
    for i, obj in enumerate(objects):
        label_coord_x.append(obj[0][0][0])
        label_coord_y.append(obj[0][0][1])

        finding_tips_img = np.zeros(segmented_img.shape[:2], np.uint8)
        cv2.drawContours(finding_tips_img, objects, i, (255, 255, 255), 1, lineType=8)

        segment_tips = find_tips(finding_tips_img)
        tip_objects, _ = _cv2_findcontours(bin_img=segment_tips)

        points = []
        for t in tip_objects:
            x, y = t.ravel()
            points.append((x, y))

        cv2.line(labeled_img, points[0], points[1], rand_color[i], 1)

    # Restore debug mode
    params.debug = debug

    # Overlay curvature scores on image
    segment_ids = []
    for i, _ in enumerate(objects):
        text = f"{curvature_measure[i]:0,.3f}"
        w = label_coord_x[i]
        h = label_coord_y[i]
        cv2.putText(
            img=labeled_img, text=text, org=(w, h),
            fontFace=cv2.FONT_HERSHEY_SIMPLEX,
            fontScale=params.text_size,
            color=(150, 150, 150),
            thickness=params.text_thickness
        )
        segment_ids.append(i)

    # Save curvature measurements
    outputs.add_observation(
        sample=label,
        variable='segment_curvature',
        trait='segment curvature',
        method='plantcv.plantcv.morphology.segment_curvature',
        scale='none',
        datatype=list,
        value=curvature_measure,
        label=segment_ids
    )

    _debug(
        visual=labeled_img,
        filename=os.path.join(params.debug_outdir, f"{params.device}_segment_curvature.png")
    )

    straightLength = eu_lengths[0]
    spineLength = path_lengths[0]

    return curvature_measure[0], spineLength, straightLength



def michaels_Distance_Detector(points_r, centroid_r, bline_r, label=None):
    """
    Calculates average vertical, horizontal, and Euclidean distances from two reference points:
    1. The centroid of the object
    2. A manually defined baseline point

    Inputs:
    points_r   = ndarray of (x, y) coordinates; typically pseudolandmarks
    centroid_r = tuple (cx, cy); the centroid in rescaled coordinates
    bline_r    = tuple (bx, by); the baseline point in rescaled coordinates
    label      = optional label string for identifying the observation outputs

    Returns:
    hori_ave_b = average horizontal distance from baseline point
    """

    # Set default label if not provided
    if label is None:
        label = params.sample_label

    # Increment device for PlantCV internal state tracking
    params.device += 1

    # Initialize distance storage lists
    vert_dist_c = []
    hori_dist_c = []
    euc_dist_c = []

    # Unpack centroid coordinates
    cx, cy = centroid_r

    # Ensure centroid is numeric (if not, return 'NA' placeholders)
    if not isinstance(cy, numbers.Number):
        return ('NA', 'NA'), ('NA', 'NA'), ('NA', 'NA'), ('NA', 'NA'), ('NA', 'NA'), ('NA', 'NA'), ('NA', 'NA'), ('NA', 'NA')

    # Loop over all points to calculate distances from centroid
    for pt in points_r:
        x, y = pt
        vert_dist_c.append(y - cy)                        # vertical distance (can be negative)
        hori_dist_c.append(abs(x - cx))                   # horizontal distance (absolute)
        euc_dist_c.append(np.sqrt((cx - x)**2 + (cy - y)**2))  # Euclidean distance

    # Compute means of distances from centroid
    vert_ave_c = np.mean(vert_dist_c)
    hori_ave_c = np.mean(hori_dist_c)
    euc_ave_c = np.mean(euc_dist_c)

    # Repeat the process for the baseline point
    vert_dist_b = []
    hori_dist_b = []
    euc_dist_b = []

    bx, by = bline_r

    for pt in points_r:
        x, y = pt
        vert_dist_b.append(y - by)
        hori_dist_b.append(abs(x - bx))
        euc_dist_b.append(np.sqrt((bx - x)**2 + (by - y)**2))

    vert_ave_b = np.mean(vert_dist_b)
    hori_ave_b = np.mean(hori_dist_b)
    euc_ave_b = np.mean(euc_dist_b)

    # Log all computed averages to the outputs module
    outputs.add_observation(sample=label, variable='vert_ave_c',
                            trait='average vertical distance from centroid',
                            method='plantcv.plantcv.landmark_reference_pt_dist',
                            scale='pixels', datatype=float, value=vert_ave_c, label='pixels')

    outputs.add_observation(sample=label, variable='hori_ave_c',
                            trait='average horizontal distance from centroid',
                            method='plantcv.plantcv.landmark_reference_pt_dist',
                            scale='pixels', datatype=float, value=hori_ave_c, label='pixels')

    outputs.add_observation(sample=label, variable='euc_ave_c',
                            trait='average Euclidean distance from centroid',
                            method='plantcv.plantcv.landmark_reference_pt_dist',
                            scale='pixels', datatype=float, value=euc_ave_c, label='pixels')

    outputs.add_observation(sample=label, variable='vert_ave_b',
                            trait='average vertical distance from baseline',
                            method='plantcv.plantcv.landmark_reference_pt_dist',
                            scale='pixels', datatype=float, value=vert_ave_b, label='pixels')

    outputs.add_observation(sample=label, variable='hori_ave_b',
                            trait='average horizontal distance from baseline',
                            method='plantcv.plantcv.landmark_reference_pt_dist',
                            scale='pixels', datatype=float, value=hori_ave_b, label='pixels')

    outputs.add_observation(sample=label, variable='euc_ave_b',
                            trait='average Euclidean distance from baseline',
                            method='plantcv.plantcv.landmark_reference_pt_dist',
                            scale='pixels', datatype=float, value=euc_ave_b, label='pixels')

    # Return average horizontal distance from the baseline
    return hori_ave_b



def calculate_rotation_angle(cx, cy, bx, by):
    """
    Calculate the angle (in degrees) between the centroid and baseline point.

    The angle returned is relative to vertical axis (Y-axis), useful for rotation correction.

    Inputs:
    cx, cy = coordinates of centroid
    bx, by = coordinates of baseline point

    Returns:
    angle_deg = angle in degrees (float)
    """
    zx = bx - cx
    zy = by - cy

    angle_rad = math.atan2(zx, zy)  # Notice the order: atan2(x, y) for rotation relative to Y-axis
    angle_deg = math.degrees(angle_rad)
    return angle_deg



def michaels_ReadAndRotate3(image, unprocessed_image):
    """
    Aligns image using the gpty_axis_pseudolandmarks function to make the primary axis vertical.
    Uses point 0 (top) and point 49 (bottom) to define the axis.

    Parameters:
    image              = the binary mask of the object
    unprocessed_image  = the original grayscale or color image for landmark detection

    Returns:
    curvaturemeasure3  = horizontal curvature metric from Distance_Detector
    finalTopPoint2     = rotated top landmark
    finalBottomPoint2  = rotated bottom landmark
    cumulative_diff2   = total X-difference between successive landmarks
    cx1, cy1, bx1, by1 = original coordinates of top and bottom before rotation
    """

    center_h = gpty_axis_pseudolandmarks(img=unprocessed_image, mask=image)
    center_landmarks1 = pcv.outputs.observations['default']['center_h_lmk']['value']

    centroid_r = center_landmarks1[0]
    bline_r = center_landmarks1[49]

    cx, cy = centroid_r
    bx, by = bline_r

    # Save original unrotated axis points
    cx1, cy1 = centroid_r
    bx1, by1 = bline_r

    if cx != bx:
        counter = 1

        while cx != bx:
            angle_deg = calculate_rotation_angle(cx, cy, bx, by)

            # Occasionally increase angle magnitude (possibly unnecessary?)
            if counter % 10 == 0:
                angle_deg *= 5  

            # Rotate both images
            rotate_img = rotate(image, -angle_deg, True, cx, cy)
            rotate_unimg = rotate(unprocessed_image, -angle_deg, True, cx, cy)

            # Recalculate landmarks after rotation
            center_h = gpty_axis_pseudolandmarks(img=rotate_unimg, mask=rotate_img)
            center_landmarks2 = pcv.outputs.observations['default']['center_h_lmk']['value']

            centroid_r2 = center_landmarks2[0]
            bline_r2 = center_landmarks2[49]

            # Update for next loop iteration
            image = rotate_img
            unprocessed_image = rotate_unimg
            cx, cy = centroid_r2
            bx, by = bline_r2

            counter += 1

        finalTopPoint2 = center_landmarks2[0]
        finalBottomPoint2 = center_landmarks2[49]
        curvaturemeasure3 = michaels_Distance_Detector(center_landmarks2, finalTopPoint2, finalBottomPoint2)
        cumulative_diff2 = cumulative_x_difference(center_landmarks2)

        return curvaturemeasure3, finalTopPoint2, finalBottomPoint2, cumulative_diff2, cx1, cy1, bx1, by1

    else:
        finalTopPoint2 = center_landmarks1[0]
        finalBottomPoint2 = center_landmarks1[49]
        curvaturemeasure3 = michaels_Distance_Detector(center_landmarks1, finalTopPoint2, finalBottomPoint2)
        cumulative_diff2 = cumulative_x_difference(center_landmarks1)

        return curvaturemeasure3, finalTopPoint2, finalBottomPoint2, cumulative_diff2, cx1, cy1, bx1, by1

    
    
def michaels_ReadAndRotate4(image, unprocessed_image):
    """
    Aligns image using the shorty_axis_pseudolandmarks function to make the primary axis vertical.
    Uses point 0 (top) and point 19 (bottom) to define the axis.

    Parameters:
    image              = the binary mask of the object
    unprocessed_image  = the original grayscale or color image for landmark detection

    Returns:
    curvaturemeasure4  = horizontal curvature metric from Distance_Detector
    finalTopPoint3     = rotated top landmark
    finalBottomPoint3  = rotated bottom landmark
    cumulative_diff1   = total X-difference between successive landmarks
    cx1, cy1, bx1, by1 = original coordinates of top and bottom before rotation
    """

    center_h = shorty_axis_pseudolandmarks(img=unprocessed_image, mask=image)
    center_landmarks1 = pcv.outputs.observations['default']['center_h_lmk']['value']
    #Make the top and bottom half analysis
    top_half_landmarks = center_landmarks1[:10]
    bottom_half_landmarks = center_landmarks1[10:]
    centroid_r = center_landmarks1[0]
    bline_r = center_landmarks1[19]
    

    cx, cy = centroid_r
    bx, by = bline_r

    # Save original unrotated axis points
    cx1, cy1 = centroid_r
    bx1, by1 = bline_r

    if cx != bx:
        counter = 1

        while cx != bx:
            angle_deg = calculate_rotation_angle(cx, cy, bx, by)

            if counter % 10 == 0:
                angle_deg *= 5 

            rotate_img = rotate(image, -angle_deg, True, cx, cy)
            rotate_unimg = rotate(unprocessed_image, -angle_deg, True, cx, cy)

            center_h = shorty_axis_pseudolandmarks(img=rotate_unimg, mask=rotate_img)
            center_landmarks2 = pcv.outputs.observations['default']['center_h_lmk']['value']
            #Making the top and bottom half analysis
            top_half_landmarks = center_landmarks2[:10]
            bottom_half_landmarks = center_landmarks2[10:]
            centroid_r2 = center_landmarks2[0]
            bline_r2 = center_landmarks2[19]

            image = rotate_img
            unprocessed_image = rotate_unimg
            cx, cy = centroid_r2
            bx, by = bline_r2

            counter += 1

        finalTopPoint3 = center_landmarks2[0]
        finalBottomPoint3 = center_landmarks2[19]
        curvaturemeasure4 = michaels_Distance_Detector(center_landmarks2, finalTopPoint3, finalBottomPoint3, label = None)
        cumulative_diff1 = cumulative_x_difference(center_landmarks2)
        #Making top and bottom half analysis
        tophalf_cumulative_diff1 = cumulative_x_difference(top_half_landmarks)
        bottomhalf_cumulative_diff1 = cumulative_x_difference(bottom_half_landmarks)

        return curvaturemeasure4, finalTopPoint3, finalBottomPoint3, cumulative_diff1, tophalf_cumulative_diff1, bottomhalf_cumulative_diff1, cx1, cy1, bx1, by1

    else:
        finalTopPoint3 = center_landmarks1[0]
        finalBottomPoint3 = center_landmarks1[19]
        curvaturemeasure4 = michaels_Distance_Detector(center_landmarks1, finalTopPoint3, finalBottomPoint3, label = None)
        cumulative_diff1 = cumulative_x_difference(center_landmarks1)
        tophalf_cumulative_diff1 = cumulative_x_difference(top_half_landmarks)
        bottomhalf_cumulative_diff1 = cumulative_x_difference(bottom_half_landmarks)
        
        return curvaturemeasure4, finalTopPoint3, finalBottomPoint3, cumulative_diff1, tophalf_cumulative_diff1, bottomhalf_cumulative_diff1, cx1, cy1, bx1, by1

    
    
    
def cumulative_x_difference(spine_landmarks):
    """
    Calculate the cumulative difference in x-values between consecutive points of the spine.

    Inputs:
    spine_landmarks: List of spine landmarks where each landmark is a tuple (x, y).

    Returns:
    cumulative_diff: Cumulative difference in x-values.

    :param spine_landmarks: list
    :return cumulative_diff: float
    """
    cumulative_diff = 0
    for i in range(len(spine_landmarks) - 1):
        x1, _ = spine_landmarks[i]
        x2, _ = spine_landmarks[i + 1]
        cumulative_diff += (abs(x2 - x1))
    return cumulative_diff


In [19]:
import glob
#pcv.params.sample_label = "spine"
#pcv.params.debug = "plot" 

# Initialize dummy dataframe
# --------------------------
df = pd.DataFrame(data=np.array([['1.333.4.55548', 32.57,'[2,2]','[2,2]',32.57,60.66,60.66,1,1,1,1, 32.57,'[2,2]','[2,2]',32.57,1,1,1,1], ['4.123.225.55', 32.57,'[2,2]','[2,2]',32.57,60.66,60.66,1,1,1,1, 32.57,'[2,2]','[2,2]',32.57,1,1,1,1]]), columns=['image_name', 'curvaturemeasure4','finalTopPoint','finalBottomPoint','cumulative1', 'tophalfcumulative1', 'bottomhalfcumulative1','ogtopx','ogtopy','ogbottomx','ogbottomy', 'curvature_method_3','finalTopPoint3','finalBottomPoint3','cumulative2','ogtopx2','ogtopy2','ogbottomx2','ogbottomy2'])
df = df.set_index('image_name')


# Directory configuration
# --------------------------
directory = '/work2/09601/mz868/frontera/960dicominterpretations/'#'/work2/09601/mz868/frontera/860interpretations/'#/work2/09601/mz868/frontera/860interpretations2/'#change for local directory'/scratch1/09601/mz868/unsegmentedImages/segmentedTestImages''/work2/09601/mz868/frontera/960interpretations/'/work2/09601/mz868/frontera/820duplicates/'/work2/09601/mz868/frontera/860interpretations/
directoryjpg = '/scratch1/09601/mz868/unsegmentedImages/dicom960/'#'/scratch1/09601/mz868/unsegmentedImages/dicom8201/'#'/corral/utexas/UKB-Imaging-Genetics/unzipped_DXA_Images/Full_Body_Transparent_HPE/816_288/'#960_384
#start=time.time()

# Image Processing Loop
# --------------------------
for img_name in glob.glob(directory + '/*.png'):
    #pcv.params.debug = "plot"
    png_img_name = os.path.splitext(os.path.basename(img_name))[0]
    jpg_img_path = os.path.join(directoryjpg, png_img_name + '.jpg')
    if os.path.exists(jpg_img_path):
        unprocessed_image = cv2.imread(jpg_img_path, 0)
        image = cv2.imread(img_name, 0)
        name=img_name[53:-4]#change for local directory
        try:#having to change to curvature_method_2 from curvaturemeasure 6/9/25
            curvaturemeasure,finalTopPoint,finalBottomPoint,cumulative1, tophalfcumulative1, bottomhalfcumulative1, ogtopx,ogtopy,ogbottomx,ogbottomy = michaels_ReadAndRotate4(image,unprocessed_image)
            
            #curvaturemeasure3,finalTopPoint2,finalBottomPoint2,cumulative2,ogtopx2,ogtopy2,ogbottomx2,ogbottomy2 = michaels_ReadAndRotate3(image,unprocessed_image)
        
        except RuntimeError:
            print(name+" raised a runtime error")
            continue
        except IndexError:
            print(name+" raised an index error")
            continue
        #having to change to curvature_method_2 from curvaturemeasure 6/9/25
    df.loc[str(name)] = [curvaturemeasure,finalTopPoint,finalBottomPoint,cumulative1, tophalfcumulative1, bottomhalfcumulative1, ogtopx,ogtopy,ogbottomx,ogbottomy]
    #df.loc[str(name)] = [curvaturemeasure4,finalTopPoint,finalBottomPoint,cumulative1, tophalfcumulative1, bottomhalfcumulative1, ogtopx,ogtopy,ogbottomx,ogbottomy,curvaturemeasure3,finalTopPoint2,finalBottomPoint2,cumulative2,ogtopx2,ogtopy2,ogbottomx2,ogbottomy2]

print(df)
df.to_csv('6-9-25fr0ntera960dicom.csv', index=True) 







#df = pd.DataFrame(data=np.array([['1.333.4.55548', 32.57,'[2,2]','[2,2]',32.57,1,1,1,1, 32.57,'[2,2]','[2,2]',32.57,1,1,1,1], ['4.123.225.55', 32.57,'[2,2]','[2,2]',32.57,1,1,1,1, 32.57,'[2,2]','[2,2]',32.57,1,1,1,1]]), columns=['image_name', 'curvature_method_2','finalTopPoint','finalBottomPoint','cumulative1','ogtopx','ogtopy','ogbottomx','ogbottomy', 'curvature_method_3','finalTopPoint3','finalBottomPoint3','cumulative2','ogtopx2','ogtopy2','ogbottomx2','ogbottomy2'])


#print(df)
#df = df.set_index('image_name')
#print(df)
#directory = '/work2/09601/mz868/frontera/860dicominterpretations2/'#'/work2/09601/mz868/frontera/860interpretations/'#/work2/09601/mz868/frontera/860interpretations2/'#change for local directory'/scratch1/09601/mz868/unsegmentedImages/segmentedTestImages''/work2/09601/mz868/frontera/960interpretations/'/work2/09601/mz868/frontera/820duplicates/'/work2/09601/mz868/frontera/860interpretations/
#directoryjpg = '/scratch1/09601/mz868/unsegmentedImages/dicom820/'#'/corral/utexas/UKB-Imaging-Genetics/unzipped_DXA_Images/Full_Body_Transparent_HPE/816_288/'#960_384
#start=time.time()
#for img_name in glob.glob(directory + '/*.png'):
    #pcv.params.debug = "plot"
    #png_img_name = os.path.splitext(os.path.basename(img_name))[0]
    #jpg_img_path = os.path.join(directoryjpg, png_img_name + '.jpg')
    #if os.path.exists(jpg_img_path):
        #unprocessed_image = cv2.imread(jpg_img_path, 0)
        #image = cv2.imread(img_name, 0)
        #name=img_name[53:-4]#change for local directory
        #try:
        #with the skeletonization
            #print(img_name)
            #curvaturemeasure,finalTopPoint,finalBottomPoint,cumulative1,ogtopx,ogtopy,ogbottomx,ogbottomy = michaels_ReadAndRotate4(image,unprocessed_image)
        #curvature_method_2shortened,finalTopPoint2,finalBottomPoint2 = michaels_ReadAndRotate2(image) #with the shortened length(yhomologyshortened)
        #without skeletonization
            #curvaturemeasure3,finalTopPoint2,finalBottomPoint2,cumulative2,ogtopx2,ogtopy2,ogbottomx2,ogbottomy2 = michaels_ReadAndRotate3(image,unprocessed_image)#using the mask instead of skeleton
        #curvature_method_2shortened,finalTopPoint2,finalBottomPoint2 = michaels_ReadAndRotate4(image)#using mask instead of skeleton and shortened
        #except RuntimeError:
            #print(name+" raised a runtime error")
            #continue
        #except IndexError:
            #print(name+" raised an index error")
            #continue
        
    #df.loc[str(name)] = [curvaturemeasure,finalTopPoint,finalBottomPoint,cumulative1,ogtopx,ogtopy,ogbottomx,ogbottomy,curvaturemeasure3,finalTopPoint2,finalBottomPoint2,cumulative2,ogtopx2,ogtopy2,ogbottomx2,ogbottomy2]
    #df.loc[str(name)] = [curvaturemeasure2,finalTopPoint,finalBottomPoint,curvature_method_2shortened,finalTopPoint2,finalBottomPoint2, curvature_method_3,finalTopPoint3,finalBottomPoint3, curvature_method_3shortened,finalTopPoint4,finalBottomPoint4]

#print(df)
#df.to_csv('fr0ntera820_2dicom.csv', index=True) 





#end = time.time()
#print("method2:")
#print(end - start)

              curvaturemeasure4 finalTopPoint finalBottomPoint cumulative1  \
image_name                                                                   
1.333.4.55548             32.57         [2,2]            [2,2]       32.57   
4.123.225.55              32.57         [2,2]            [2,2]       32.57   

              tophalfcumulative1 bottomhalfcumulative1 ogtopx ogtopy  \
image_name                                                             
1.333.4.55548              60.66                 60.66      1      1   
4.123.225.55               60.66                 60.66      1      1   

              ogbottomx ogbottomy curvature_method_3 finalTopPoint3  \
image_name                                                            
1.333.4.55548         1         1              32.57          [2,2]   
4.123.225.55          1         1              32.57          [2,2]   

              finalBottomPoint3 cumulative2 ogtopx2 ogtopy2 ogbottomx2  \
image_name                             